# BZA cases by proposed use

Reuse the same point-map API with a different categorical encoding.

This is an editable worked example of [`bza_proposed_use_map.py`](../src/graphics/bza_proposed_use_map.py). It runs Python SDK calls directly—no website, CLI subprocess, or registered build dispatcher. The canonical definition remains the publishing source of truth.

**What the numbers mean:** Proposed-use families are analytic groupings, not literal zoning determinations. Only located cases are mapped. If you filter cases, recalculate the residential share before keeping the existing headline.

Start with **Run All**, inspect the data table and preview, then change the title or a visual encoding in step 4. Parcel examples load the full city and can take several minutes.


## 1. Open the libraries

Use the environment in [README.md](README.md). Paths below locate the checkout, not a personal machine.


In [ ]:
from pathlib import Path
import sys

# Run from this notebook folder or anywhere inside the Detroit checkout.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "strongtowns-data.lock.json").is_file()
             and (p / "projects/graphics/src/graphics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter inside the strongtowns-detroit checkout; see README.md.")
sys.path.insert(0, str(ROOT / "src"))
GRAPHICS = ROOT / "projects/graphics"
FORUM = ROOT / "projects/detroit-land-use-forum"
import strongtowns_graphics as graphics_sdk
if not hasattr(graphics_sdk, "GraphicInput"):
    raise RuntimeError(
        "This kernel has an older graphics SDK. Restart Jupyter with the uv command "
        "in README.md so it uses this project's locked dependencies."
    )
from IPython.display import SVG, display
from strongtowns_graphics import (
    GraphicFormat, GRAPHIC_FORMAT_SPECS, render_graphic_canvas,
    render_graphic_svg, write_graphic_bundle,
)

NOTEBOOK_NAME = 'bza_proposed_use_map'


In [ ]:
import sys

from pathlib import Path

import geopandas as gpd

import pandas as pd

import polars as pl

SOURCE_DIR = FORUM / "bza-use-types"

sys.path.insert(0, str(SOURCE_DIR))

sys.path.insert(0, str(GRAPHICS))

from build_use_type_assets import (  # noqa: E402
    FAMILY_COLORS,
    selected_cases,
)

from basemap import load_detroit_basemap  # noqa: E402

from strongtowns_graphics import (
    GraphicInput,
    LegendOrders,
    MapMarkerStyle,
    MobileMapInset,
    MobileMapPocket,
    categorical_proportional_symbol_map,
    graphic_definition,
)


## 2. Resolve the prepared data

The aliases below name the inputs you will read. The data SDK resolves only the snapshots pinned by this project. Change prepared inputs through a reviewed data lock update, not by pointing at a moving latest file.


In [ ]:
from strongtowns_data import DataBuildSystem, DataLock, DataRepository
from strongtowns_detroit.repositories import data_repository
from strongtowns_graphics import GraphicBuildContext

requirements = (
        GraphicInput(
            "classifications",
            "detroit.bza.gemini.raw",
            "raw/project_type_enrichment/case_histories_with_project_types.csv",
        ),
        GraphicInput("sites", "detroit.bza.gemini.raw", "raw/map_sites.gpkg"),
        GraphicInput("boundary", "detroit.osm.basemap.raw", "detroit_boundary.geojson"),
        GraphicInput("water", "detroit.osm.basemap.raw", "detroit_water.geojson"),
        GraphicInput("roads", "detroit.base-units.streets.raw", "raw.geojson"),
    )
lock = DataLock.load(ROOT / "strongtowns-data.lock.json")
repository = DataRepository(DataBuildSystem.find(data_repository()))
paths, provenance = {}, {}
for requirement in requirements:
    try:
        reference = lock.asset(requirement.dataset_id)
        paths[requirement.alias] = repository.artifact(reference, requirement.artifact)
        if not paths[requirement.alias].is_file():
            raise FileNotFoundError(paths[requirement.alias])
        provenance[requirement.alias] = {
            "dataset": requirement.dataset_id,
            "artifact": requirement.artifact,
            "snapshot": reference.snapshot_id,
            "manifest_sha256": reference.manifest_sha256,
        }
    except (ValueError, KeyError, FileNotFoundError) as error:
        raise RuntimeError(
            f"Prepared input unavailable: {requirement.dataset_id}/{requirement.artifact}. "
            "Ask the data maintainer to restore the pinned snapshot in strongtowns-data; "
            "this notebook never fetches data or changes the lock."
        ) from error
context = GraphicBuildContext(paths)
provenance


## 3. Prepare and inspect the table

This follows the existing graphic’s data selection and calculations. The displayed rows are a preview; the graphic uses the full prepared table.


In [ ]:
cases = selected_cases(pd.read_csv(context.input("classifications")))

sites = gpd.read_file(context.input("sites")).to_crs("EPSG:3857")

categories = {
    key: (label, FAMILY_COLORS[key])
    for key, label in (
        ("housing", "Residential projects"),
        ("cannabis_or_controlled_use", "Cannabis or controlled use"),
        ("vehicle_oriented", "Vehicle sales and services"),
        ("mixed_use", "Mixed-use"),
        ("retail_or_personal_service", "Retail or personal service"),
        ("food_or_beverage", "Food or beverage"),
        ("institutional_or_civic", "Institutional or civic"),
        ("parking_only", "Parking"),
        ("industrial_or_logistics", "Industrial or logistics"),
        ("signage", "Signage"),
        ("office_or_medical", "Office or medical"),
        ("recreation_or_open_space", "Recreation or open space"),
        ("other", "Other / insufficient detail"),
        ("religious", "Religious"),
    )
}

case_categories = cases.set_index("case_history_id")["display_family"]

appearance_counts = cases.set_index("case_history_id")["appearance_count"]

located = sites[
    sites["case_history_id"].isin(cases["case_history_id"])
].drop_duplicates(["case_history_id", "site_id", "parcel_id"]).copy()

located["category"] = located["case_history_id"].map(case_categories)

located["magnitude"] = (
    located["case_history_id"].map(appearance_counts).fillna(1).astype(int)
)

dissolved = located.dissolve(by=["case_history_id", "category"])

points = dissolved.geometry.representative_point()

records = []

for (case_id, category), point in points.items():
    label, color = categories[category]
    records.append(
        {
            "point_id": str(case_id),
            "easting": point.x,
            "northing": point.y,
            "category": category,
            "category_label": label,
            "color": color,
            "magnitude": int(dissolved.loc[(case_id, category), "magnitude"]),
            "category_count": 1,
        }
    )


In [ ]:
display(pl.DataFrame(records).head(8))


## 4. Build the graphic with the SDK

Edit `title`, `subtitle`, colors, legends, or explicit encodings here. Keep sources and descriptions accurate when changing data. The parcel and travel examples reuse existing project map-drawing helpers, then call SDK composition functions; the bar, point-map, and continuous-choropleth examples expose their renderer calls directly.


In [ ]:
graphic = categorical_proportional_symbol_map(
    pl.DataFrame(records),
    basemap=load_detroit_basemap(
        context.input("boundary"), context.input("roads"), context.input("water")
    ),
    title="Over one quarter of Detroit's BZA cases involve residential projects",
    subtitle=(
        "Cases grouped by the project described in meeting minutes, 2019–2026"
    ),
    category_legend_heading="Proposed use",
    magnitude_legend_heading="HEARINGS PER CASE",
    category_order=LegendOrders.DESCENDING,
    marker_style=MapMarkerStyle(opacity=0.78, overlap_fraction=0.10),
    sources=(
        "Source: Detroit BZA minutes, 2019–2026; locations linked to City "
        "assessor parcels. Proposed-use labels are analytic groupings derived from the minutes. "
        "To aid legibility, locations may not represent precise addresses.",
    ),
    description=(
        f"A map of {len(records)} located Detroit BZA case histories grouped "
        "by the proposed use described in meeting minutes."
    ),
)

graphics = {"bza-proposed-use-map": graphic}


## 5. Choose the Instagram format and preview

Feed uses 1080 × 1350; story uses 1080 × 1920 with the library’s established content padding. Changing this enum preserves the publishing policy.


In [ ]:
# Change to GraphicFormat.INSTAGRAM_STORY for a story-sized export.
TARGET = GraphicFormat.INSTAGRAM_POST
EXPORT_PNG = True  # SVG and HTML work without the rsvg-convert system tool.
target = GRAPHIC_FORMAT_SPECS[TARGET]


In [ ]:
# Preview exactly the composition used by the export below.
for name, graphic in graphics.items():
    print(name)
    svg = (render_graphic_canvas(
        graphic, canvas_aspect_ratio=target.aspect_ratio,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding,
    ) if target.content_aspect_ratio else render_graphic_svg(
        graphic, aspect_ratio=target.aspect_ratio,
    ))
    display(SVG(svg))
    print("Alt text:", graphic.description or graphic.title_text)


## 6. Export

PNG is ready for Instagram; SVG and HTML retain the composition for inspection. Copy the adjacent alt-text file when posting. Exports go to the ignored `projects/graphics/output/notebooks/` directory and can be regenerated. Inspect all pages before sharing.


In [ ]:
import json
import shutil

output_dir = GRAPHICS / "output/notebooks" / NOTEBOOK_NAME / TARGET.value
formats = ("html", "svg", "png") if EXPORT_PNG else ("html", "svg")
if EXPORT_PNG and shutil.which("rsvg-convert") is None:
    raise RuntimeError(
        "PNG export needs rsvg-convert (see README.md). "
        "Set EXPORT_PNG = False above and rerun the export to save SVG/HTML now."
    )
for name, graphic in graphics.items():
    files = write_graphic_bundle(
        output_dir, name, graphic,
        aspect_ratio=target.aspect_ratio, png_width=target.png_width,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding, formats=formats,
    )
    (output_dir / f"{name}.alt.txt").write_text(
        graphic.description or graphic.title_text, encoding="utf-8"
    )
    for kind, path in files.items():
        print(f"{kind}: {path}")
# Keep the exact source identities alongside your exported graphics.
(output_dir / "sources.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8"
)


## Try the pattern on another question

Make a copy of this notebook. Start by changing editorial wording, then inspect the explicit input table before changing a field or grouping. Keep units, unknown records, source coverage, and denominators visible. Changing geographic scope or a legal threshold requires reviewing the method and claim, not just replacing the title.

Use the other notebooks to compare stacked bars, categorized points, continuous parcel maps, and mobile map compositions. Clear outputs before committing a notebook; put publishing changes back into the canonical definition.
